# Fine-tuning을 이용한 AI에게 인격 부여하기
## OpenAI Fine-tuning API 학습을 위한 데이터셋 만들기

- jsonline 포맷
- 예시
```json
{"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "What's the capital of France?"}, {"role": "assistant", "content": "Paris, as if everyone doesn't know that already."}]}
{"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "Who wrote 'Romeo and Juliet'?"}, {"role": "assistant", "content": "Oh, just some guy named William Shakespeare. Ever heard of him?"}]}
{"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "How far is the Moon from Earth?"}, {"role": "assistant", "content": "Around 384,400 kilometers. Give or take a few, like that really matters."}]}
```

In [3]:
from operator import itemgetter
import json

from tqdm.notebook import tqdm
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI

## 데이터 생성

In [4]:
model = ChatOpenAI(model='gpt-3.5-turbo', temperature=0.8)

In [5]:
ai_1_system_prompt = f"""\
- 입국심사를 받으러 공항에 온 사람이다.
- 입국심사관이 요구하는 대로 응답해라.
- 너는 영어로 응답한다.
"""

ai_2_system_prompt = f"""\
- 너는 출입국 사무소의 직원이다.
- 아래의 단계로 질문을 한다.
1. 이름 묻기
2. 여권 제시 요구하기
3. 여행의 목적 묻기
4. 몇일간 체류하는지 묻기
5. 모든 질문에 답이 끝났으면 [END] 라고 이야기한다.
- 너는 영어로 응답한다.\
"""


def get_new_ai_chains():
    ai_1_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", ai_1_system_prompt),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
        ]
    )
    ai_1_memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    ai_1_chain = (
        RunnablePassthrough.assign(
            chat_history=RunnableLambda(ai_1_memory.load_memory_variables) | itemgetter("chat_history")
        )
        | ai_1_prompt
        | model
    )
    
    
    ai_2_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", ai_2_system_prompt),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
        ]
    )
    ai_2_memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    ai_2_chain = (
        RunnablePassthrough.assign(
            chat_history=RunnableLambda(ai_2_memory.load_memory_variables) | itemgetter("chat_history")
        )
        | ai_2_prompt
        | model
    )
    return ai_1_chain, ai_1_memory, ai_2_chain, ai_2_memory

In [6]:
conversation_list = []

n_conversation = 130
n_max_turn = 20

for _ in tqdm(range(n_conversation), total=n_conversation):
    ai_1_chain, ai_1_memory, ai_2_chain, ai_2_memory = get_new_ai_chains()
    ai_2_output = model.invoke("Start a conversation as a immigration officer to screen ㅁ entrant in a single sentence").content
    
    print("*"*10 + "새로운 대화" + "*"*30)
    print(f"입국 심사관: {ai_2_output}")
    for _ in range(n_max_turn):
        ai_1_output =  ai_1_chain.invoke({"input": ai_2_output}).content
        ai_1_memory.save_context({"input": ai_2_output}, {"output": ai_1_output})
        print(f"여행객: {ai_1_output}")
    
        ai_2_output =  ai_2_chain.invoke({"input": ai_1_output}).content
        ai_2_memory.save_context({"input": ai_1_output}, {"output": ai_2_output})
        print(f"입국 심사관: {ai_2_output}")
        if "[END]" in ai_2_output:
            break
    conversation_list.append(ai_2_memory.chat_memory.dict()['messages'])

  0%|          | 0/130 [00:00<?, ?it/s]

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_21628\3217185967.py:27: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  ai_1_memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


**********새로운 대화******************************
입국 심사관: Good afternoon, can you please state your reason for entering the country and provide your passport for verification?
여행객: Hello, I am here for tourism purposes. Here is my passport for verification.
입국 심사관: 1. May I have your name, please?
2. Could you please show me your passport for verification?
3. What is the purpose of your trip?
4. How many days do you plan to stay?
5. Once all the questions are answered, I will say [END].


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_21628\97581343.py:22: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  conversation_list.append(ai_2_memory.chat_memory.dict()['messages'])


**********새로운 대화******************************
입국 심사관: Hello, can you please provide me with your passport and visa information for entry into the country?
여행객: Sure, here is my passport. My passport number is ABC123456 and my visa number is XYZ789012.
입국 심사관: 1. What is your name?
2. May I see your passport, please?
3. What is the purpose of your travel?
4. How many days will you be staying?
5. [END]


...

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_21628\97581343.py:22: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  conversation_list.append(ai_2_memory.chat_memory.dict()['messages'])


**********새로운 대화******************************
입국 심사관: Good morning, can you please provide me with your passport and travel documents for inspection?
여행객: Good morning. Sure, here are my passport and travel documents for inspection.
입국 심사관: Good morning. May I have your name, please?
여행객: Good morning. My name is [insert your name].
입국 심사관: Thank you. Can I see your passport, please?
여행객: Of course. Here is my passport.
입국 심사관: What is the purpose of your trip?
여행객: I am here for tourism.
입국 심사관: How many days will you be staying in the country?
여행객: I will be staying for 10 days.
입국 심사관: [END]


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_21628\97581343.py:22: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  conversation_list.append(ai_2_memory.chat_memory.dict()['messages'])


**********새로운 대화******************************
입국 심사관: Good afternoon, can you please provide me with your passport and visa documentation for entry into the country?
여행객: Sure, here are my passport and visa documents for entry.
입국 심사관: 1. What is your name?
2. May I see your passport, please?
3. What is the purpose of your trip?
4. How many days will you be staying?
5. [END]


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_21628\97581343.py:22: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  conversation_list.append(ai_2_memory.chat_memory.dict()['messages'])


In [7]:
conversation_list[0][:3]

[{'content': 'Hello, I am here for tourism purposes. Here is my passport for verification.',
  'additional_kwargs': {},
  'response_metadata': {},
  'type': 'human',
  'name': None,
  'id': None},
 {'content': '1. May I have your name, please?\n2. Could you please show me your passport for verification?\n3. What is the purpose of your trip?\n4. How many days do you plan to stay?\n5. Once all the questions are answered, I will say [END].',
  'additional_kwargs': {},
  'response_metadata': {},
  'type': 'ai',
  'name': None,
  'id': None}]

## OpenAI 포맷으로 변환

In [8]:
def convert_message(msg):
    new_msg = {
        "role": 'user' if msg['type'] == 'human' else 'assistant',
        "content": msg['content']
    }
    return new_msg

In [7]:
# new_conversation_list = []

# system_msg = {"role": "system", "content": ai_2_system_prompt}

# for conversation in conversation_list:
#     new_conversation = [system_msg]
#     for msg in conversation[1:]:
#         new_conversation.append(convert_message(msg))
#     new_conversation_list.append(new_conversation)

In [14]:
new_conversation_list = []

for conversation in conversation_list:
    new_conversation = []
    for msg in conversation[1:]:
        new_conversation.append(convert_message(msg))
    new_conversation_list.append({"messages" : new_conversation})

In [15]:
new_conversation_list

[{'messages': [{'role': 'assistant',
    'content': '1. May I have your name, please?\n2. Could you please show me your passport for verification?\n3. What is the purpose of your trip?\n4. How many days do you plan to stay?\n5. Once all the questions are answered, I will say [END].'}]},
 {'messages': [{'role': 'assistant',
    'content': '1. What is your name?\n2. May I see your passport, please?\n3. What is the purpose of your travel?\n4. How many days will you be staying?\n5. [END]'}]},
 {'messages': [{'role': 'assistant',
    'content': '1. What is your name?\n2. May I see your passport, please?\n3. What is the purpose of your travel?\n4. How many days will you be staying?\n5. [END]'}]},
 {'messages': [{'role': 'assistant',
    'content': '1. What is your name?\n2. May I see your passport, please?\n3. What is the purpose of your trip?\n4. How many days will you be staying?\n5. Do you have any items to declare?\n[END]'}]},
 {'messages': [{'role': 'assistant',
    'content': '1. May I

## 데이터셋 나누기

In [16]:
n_train = 100
train_dataset = new_conversation_list[:n_train]
valid_dataset = new_conversation_list[n_train:]

## 데이터 저장하기

In [17]:
def save_as_json_lines(list_of_dicts, file_name):
    with open(file_name, 'w') as file:
        for dictionary in list_of_dicts:
            json_line = json.dumps(dictionary, ensure_ascii=False)
            file.write(json_line + '\n')

In [18]:
save_as_json_lines(train_dataset, "./immigration_train.jsonl")
save_as_json_lines(valid_dataset, "./immigration_valid.jsonl")

## 더 필요한 과정

- 데이터셋 정제하기
- 더 좋은 모델로 데이터 샘플링하기